# E13 FN-PROBE-2: Qwen3.8-Flash-Next on 4xL4 (SGLang 0.5.21), Stage 1 of PLAN rev 4.1
No model is used for the submission: CELL 0 writes the identity fallback submission.json (dual-mode safe).

In [ ]:
# ==================================================================== CELL 0 — mode + fallback submission (stdlib only)
import os, sys, json, time, random, hashlib, re, shutil, subprocess, threading, traceback, zipfile, math
from pathlib import Path
from datetime import datetime, timezone
import concurrent.futures as cf

NOTEBOOK_T0 = time.perf_counter()
# S1 (CLOCK): Kaggle's 12 h counts from kernel start; R5's cell 0 began 7 s later. Anchor the clock at the kernel
# process start (capped at 10 min in case the value is odd).
try:
    import psutil as _ps
    _since_kernel_start = max(0.0, min(600.0, time.time() - _ps.Process(os.getpid()).create_time()))
except Exception:
    _since_kernel_start = 0.0
NOTEBOOK_T0 -= _since_kernel_start
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
ARC_ROOT = Path(os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
if not ARC_ROOT.exists():
    for _alt in ("/kaggle/input/arc-prize-2026-arc-agi-2",):
        if Path(_alt).exists():
            ARC_ROOT = Path(_alt)
WORKING = Path(os.environ.get("Q27X_WORKING", "/kaggle/working")); WORKING.mkdir(parents=True, exist_ok=True)
OUT = WORKING / "q27x"; OUT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING / "submission.json"

def _load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

test_challenges = _load_json(ARC_ROOT / "arc-agi_test_challenges.json")

def write_json_atomic(path, obj):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def identity_submission(challenges):
    return {tid: [{"attempt_1": q["input"], "attempt_2": q["input"]} for q in task["test"]]
            for tid, task in challenges.items()}

write_json_atomic(SUBMISSION_PATH, identity_submission(test_challenges))
print("MODE:", "COMPETITION RERUN (hidden test)" if RERUN else "COMMIT / INTERACTIVE (evaluation)",
      "| fallback submission.json:", len(test_challenges), "tasks | RUN_ID", RUN_ID,
      "| clock anchored", round(_since_kernel_start, 1), "s before cell 0")


In [ ]:
# ==================================================================== CELL 1 — E13 configuration (PLAN rev 4.1)
import glob, base64, copy, shlex, signal as _signal, urllib.request as _urlreq
E13_HARNESS = os.environ.get("E13_HARNESS") == "1"            # local CPU harness: fake server, no install, no GPU kernels
E11_HARNESS = E13_HARNESS                                     # names used by the reused E11 helpers
OUT = WORKING / "e13"; OUT.mkdir(parents=True, exist_ok=True)
T0 = time.perf_counter()
HARD_STOP_SEC = float(os.environ.get("E13_HARD_STOP_SEC", 2.5 * 3600))      # whole notebook
L1_SEC, L2_SEC = 75 * 60, 60 * 60                                            # per launch (load + gates + speed)
READY_CAP_SEC, READY_STALL_SEC = 45 * 60, 15 * 60                            # progress-based server wait
SGL_PORT = int(os.environ.get("E13_PORT", 30000)); SGL_URL = f"http://127.0.0.1:{SGL_PORT}"
SERVED = "flashnext"
REPORT = {"run_id": RUN_ID, "plan": "E13 PLAN rev 4.1", "stages": {}, "decision": {}, "launches": []}

def elapsed():
    return time.perf_counter() - T0

def left():
    return HARD_STOP_SEC - elapsed()

def stage(name, ok, **info):
    REPORT["stages"][name] = dict(ok=ok, t_sec=round(elapsed(), 1), **info)
    write_json_atomic(OUT / "e13_probe_report.json", REPORT)
    print(f"[{elapsed():7.0f}s] {name}: {'OK' if ok else 'FAIL'} " + json.dumps(info, default=str)[:600], flush=True)

def _find_dir(pattern, test):
    for d in sorted(glob.glob(pattern, recursive=True)):
        try:
            if os.path.isdir(d) and test(Path(d)):
                return Path(d)
        except Exception:
            pass
    return None

def _qc(d):
    try:
        c = json.loads((d / "config.json").read_text())
        return c.get("quantization_config") or c.get("text_config", {}).get("quantization_config") or {}
    except Exception:
        return {}

INP = Path(os.environ.get("E13_INPUT_ROOT", "/kaggle/input"))
SGL_WHEELHOUSE = _find_dir(str(INP / "**" / "*sglang-0-5-21-cu13-wheelhouse*"), lambda d: (d / "SHA256SUMS").exists())
FA2_DIR = _find_dir(str(INP / "**" / "*flash-attn-2-8-3-post1*"), lambda d: (d / "SHA256SUMS_AND_SOURCE.txt").exists())
MODEL_DIR = _find_dir(str(INP / "**" / "*intel-qwen3*8-flash-next-w4a16-autoround*" / "**"),
                      lambda d: (d / "config.json").exists() and _qc(d).get("quant_method") == "auto-round")
DRAFT_DIR = _find_dir(str(INP / "**" / "*albucino-qwen3-8-flash-next-drafter*"), lambda d: d.exists())
PENNY_DIR = _find_dir(str(INP / "**" / "*pennyroyal-v253*"), lambda d: d.exists())
FA2_WHEEL = "flash_attn-2.8.3.post1+cu13torch2.13cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
FA2_SHA = "74984f628eaa9b22de271c787d82c4ef16daffcfb9684286a7a15ff6ad41ae84"
D2_BEFORE = "135e004f7613a59aa0765cbe3d779c9bb0c459f3a199b868358759e020aaeb5d"   # = Franzen patch #3 BEFORE_SHA = 0.5.21
D2_AFTER = "47e65140e2480a8637b9ea5bb746ff77e9c09c01e448fd2056c0b485e4f3a10a"    # = Franzen AFTER_SHA = Pennyroyal wheel
# Launch 1 (PLAN 3b): Franzen-verbatim except the pre-declared memory items (NEXTN off, chunk 2048, mamba = running x 5,
# mem 0.93 so that >= 1.5 GiB stays outside the static pools, PLAN M6)
LAUNCH1 = {"tag": "L1", "mem": "0.93", "ctx": 139264, "running": 8, "chunk": 2048, "mamba_cache": 40, "nextn": False,
           "no_cuda_graph": False, "decode": None, "tilelang_off": False}
REPORT["inputs"] = {k: str(v) if v else None for k, v in dict(wheelhouse=SGL_WHEELHOUSE, fa2=FA2_DIR, model=MODEL_DIR,
                                                              draft=DRAFT_DIR, penny=PENNY_DIR).items()}
stage("0_inputs", all([SGL_WHEELHOUSE, FA2_DIR, MODEL_DIR]) or E13_HARNESS, **REPORT["inputs"])


In [ ]:
# ==================================================================== CELL 2 — install, patches, static asserts
# ---- reused verbatim from the E12 EVAL30 notebook engine cell (Kaggle-proven): install + server process helpers
class E11Fail(Exception):
    pass


def _e11_post(path, payload, timeout=30):
    req = _urlreq.Request(SGL_URL + path, data=json.dumps(payload).encode(), method="POST",
                          headers={"Content-Type": "application/json"})
    with _urlreq.urlopen(req, timeout=timeout) as r:
        return r.read()


def _e11_gpu_used():
    try:
        return [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                               capture_output=True, text=True, timeout=30).stdout.split()]
    except Exception:
        return []


def _e11_gpu_free(max_sec=180):
    t = time.perf_counter()
    while time.perf_counter() - t < max_sec:
        u = _e11_gpu_used()
        if E11_HARNESS or (u and max(u) < 1024):
            return {"free": True, "sec": round(time.perf_counter() - t), "used_mb": u}
        time.sleep(5)
    for _pat in ("sglang.launch_server", "sglang::"):
        subprocess.run(["pkill", "-9", "-f", _pat], capture_output=True)
    time.sleep(15)
    return {"free": False, "sec": round(time.perf_counter() - t), "used_mb": _e11_gpu_used(), "pkill": True}


# ---- install (= E11-03/04 C1): a venv from the wheelhouse lock, --no-deps; CUDA 13 pip runtime as CUDA_HOME; link shims
def e11_install():
    if E11_HARNESS:
        return Path(sys.executable), dict(os.environ), {"harness": True}
    shims, t0 = {}, time.perf_counter()
    venv = Path("/tmp/sglang-venv"); vpy = venv / "bin" / "python"
    # Kaggle drops '+' from dataset file names (flashinfer_jit_cache-0.6.18+cu130 -> ...0.6.18cu130); rebuild the original
    # names from SHA256SUMS as links in a find-links folder, checking the hash of every renamed wheel
    fl = Path("/tmp/e11-wheels"); fl.mkdir(parents=True, exist_ok=True)
    renamed = []
    for line in (SGL_WHEELHOUSE / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        h, name = line.split("  ", 1); name = name.strip()
        if not name.endswith(".whl"):
            continue
        s = SGL_WHEELHOUSE / name if (SGL_WHEELHOUSE / name).exists() else SGL_WHEELHOUSE / name.replace("+", "")
        if not s.exists():
            raise E11Fail(f"wheel missing from the dataset: {name}")
        if s.name != name:
            d = hashlib.sha256()
            with open(s, "rb") as f:
                for b in iter(lambda: f.read(16 * 1024 * 1024), b""):
                    d.update(b)
            if d.hexdigest() != h:
                raise E11Fail(f"sha256 mismatch for {s.name}")
            renamed.append(f"{s.name} -> {name}")
        if not (fl / name).exists():
            (fl / name).symlink_to(s)
    shims["renamed_wheels"] = renamed
    if not vpy.exists():
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv)], check=True)
        r = subprocess.run([sys.executable, "-m", "pip", "--python", str(vpy), "install", "--no-index", "--find-links", str(fl),
                            "--requirement", str(SGL_WHEELHOUSE / "requirements.lock"), "--no-deps", "--only-binary", ":all:",
                            "--no-compile", "--disable-pip-version-check", "--no-warn-conflicts", "-q"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise E11Fail("offline SGLang install failed: " + (r.stderr or r.stdout)[-1500:])
    site = Path(subprocess.run([str(vpy), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                               capture_output=True, text=True, check=True).stdout.strip())
    cuda_home = site / "nvidia" / "cu13"
    # E11-02 BUILD_PROBE: the pip CUDA runtime ships lib/libcudart.so.13 only; JIT links need libcudart.so and lib64
    if not (cuda_home / "lib" / "libcudart.so").exists():
        (cuda_home / "lib" / "libcudart.so").symlink_to("libcudart.so.13"); shims["libcudart.so"] = "created"
    if not (cuda_home / "lib64").exists():
        (cuda_home / "lib64").symlink_to("lib"); shims["lib64"] = "created"
    # E10 run 1: FlashInfer's JIT link needs -lcuda; Kaggle has only libcuda.so.1
    libcuda = Path("/tmp/e11-libcuda")
    have = [d for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/lib64/stubs", "/usr/lib/x86_64-linux-gnu", "/usr/lib64")
            if Path(d, "libcuda.so").exists()]
    if have:
        shims["libcuda.so"] = f"exists in {have[0]}"
    else:
        real = None
        try:
            for line in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, timeout=30).stdout.splitlines():
                if "libcuda.so.1" in line and "=>" in line:
                    real = line.split("=>")[-1].strip(); break
        except Exception:
            pass
        for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
            if real is None and Path(d, "libcuda.so.1").exists():
                real = str(Path(d, "libcuda.so.1"))
        if real:
            libcuda.mkdir(parents=True, exist_ok=True)
            if (libcuda / "libcuda.so").is_symlink() or (libcuda / "libcuda.so").exists():
                (libcuda / "libcuda.so").unlink()
            (libcuda / "libcuda.so").symlink_to(Path(real).resolve()); shims["libcuda.so"] = f"shim -> {real}"
        else:
            shims["libcuda.so"] = "libcuda.so.1 NOT FOUND"
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_HOME", "CUDA_PATH")}
    env.update({
        "PATH": os.pathsep.join([str(venv / "bin"), str(cuda_home / "bin"), os.environ.get("PATH", "")]),
        "CUDA_HOME": str(cuda_home), "CUDA_PATH": str(cuda_home),
        "LIBRARY_PATH": os.pathsep.join(x for x in (str(libcuda), str(cuda_home / "lib"), os.environ.get("LIBRARY_PATH", "")) if x),
        "LD_LIBRARY_PATH": os.pathsep.join(x for x in (str(cuda_home / "lib"), os.environ.get("LD_LIBRARY_PATH", "")) if x),
        "TORCH_CUDA_ARCH_LIST": "8.9", "MAX_JOBS": "4",
        "SGLANG_CACHE_DIR": "/tmp/e11-cache/sglang", "CUDA_CACHE_PATH": "/tmp/e11-cache/cuda",
        "TRITON_CACHE_DIR": "/tmp/e11-cache/triton", "TORCH_EXTENSIONS_DIR": "/tmp/e11-cache/torch_ext",
        "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false",
        "CUDA_DEVICE_ORDER": "PCI_BUS_ID", "PYTHONUNBUFFERED": "1",
    })
    shims.update(cuda_home=str(cuda_home), venv_python=str(vpy), sec=round(time.perf_counter() - t0, 1))
    return vpy, env, shims



class E11Server:
    def __init__(self, cmd, env, tag):
        self.cmd, self.env, self.tag, self.proc, self.log_f = cmd, env, tag, None, None
        self.log_path = OUT / f"sglang_server_{tag}.log"

    def start(self):
        self.log_f = open(self.log_path, "w", encoding="utf-8")
        self.t0 = time.perf_counter()
        self.proc = subprocess.Popen(self.cmd, env=self.env, stdout=self.log_f, stderr=subprocess.STDOUT,
                                     cwd=None if E11_HARNESS else "/tmp", start_new_session=True)

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def log_text(self):
        try:
            return self.log_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            return ""

    def wait_ready(self, timeout):
        while time.perf_counter() - self.t0 < timeout:
            if not self.alive():
                return False, f"server exited rc={self.proc.returncode}"
            try:
                _urlreq.urlopen(SGL_URL + "/health", timeout=10).read()
                return True, f"ready in {time.perf_counter() - self.t0:.0f}s"
            except Exception:
                time.sleep(2 if E11_HARNESS else 5)
        return False, "timeout"

    def kill(self, wait=True):
        if self.alive():
            for sig, w in ((_signal.SIGTERM, 30 if wait else 0), (getattr(_signal, "SIGKILL", _signal.SIGTERM), 30)):
                try:
                    if hasattr(os, "killpg"):
                        os.killpg(self.proc.pid, sig)          # the whole group: TP workers, scheduler, detokenizer
                    else:
                        self.proc.kill()
                except Exception:
                    pass
                try:
                    self.proc.wait(timeout=w if w else 0.1)
                    break
                except Exception:
                    continue
        if self.log_f:
            try:
                self.log_f.close()
            except Exception:
                pass
            self.log_f = None




E13_SCRIPTS = json.loads(base64.b64decode("...").decode())
SCRIPT_DIR = OUT / "scripts"; SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
for _n, _s in E13_SCRIPTS.items():
    (SCRIPT_DIR / _n).write_text(_s, encoding="utf-8")
sys.path.insert(0, str(SCRIPT_DIR))
import e13_gate

def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(16 << 20), b""):
            h.update(b)
    return h.hexdigest()

def _replace_once_atomic(path, old, new, label):
    t = path.read_text(encoding="utf-8")
    if t.count(old) != 1:
        raise E11Fail(f"{label}: anchor found {t.count(old)} times")
    tmp = path.with_name(path.name + ".e13tmp"); tmp.write_text(t.replace(old, new), encoding="utf-8")
    os.replace(tmp, path)

VPY, VENV_ENV, SITE = None, None, None
try:
    VPY, VENV_ENV, _shims = e11_install()
    SITE = Path(subprocess.run([str(VPY), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                               capture_output=True, text=True, check=True).stdout.strip())
    stage("1_install", True, shims=_shims, site=str(SITE))
    if not E13_HARNESS:
        # FA2 (R16): Kaggle may drop '+' from file names; restore the name as a link after the SHA check, then --no-deps
        _w = FA2_DIR / FA2_WHEEL if (FA2_DIR / FA2_WHEEL).exists() else FA2_DIR / FA2_WHEEL.replace("+", "")
        if _sha(_w) != FA2_SHA:
            raise E11Fail("FA2 wheel sha256 mismatch")
        _fl = Path("/tmp/e13-fa2"); _fl.mkdir(exist_ok=True)
        if not (_fl / FA2_WHEEL).exists():
            (_fl / FA2_WHEEL).symlink_to(_w)
        _r = subprocess.run([sys.executable, "-m", "pip", "--python", str(VPY), "install", "--no-index", "--no-deps",
                             "--find-links", "/tmp/e11-wheels", str(_fl / FA2_WHEEL), "-q", "--disable-pip-version-check"],
                            capture_output=True, text=True)
        if _r.returncode:
            raise E11Fail("FA2 install failed: " + (_r.stderr or _r.stdout)[-800:])
        # The FA2 wheel also ships 21 older flash_attn/cute/* files that overwrite FA4's (flash_attn_4 has no
        # flash_attn/__init__.py). Reinstall the wheelhouse FA4 so its cute files are restored; FA2's own files stay.
        _fa4 = sorted(Path("/tmp/e11-wheels").glob("flash_attn_4-*.whl"))
        if _fa4:
            _r = subprocess.run([sys.executable, "-m", "pip", "--python", str(VPY), "install", "--no-index", "--no-deps",
                                 "--force-reinstall", str(_fa4[0]), "-q", "--disable-pip-version-check"],
                                capture_output=True, text=True)
            if _r.returncode:
                raise E11Fail("FA4 restore failed: " + (_r.stderr or _r.stdout)[-800:])
        # D2 = Franzen patch #3: two substitutions, his own BEFORE/AFTER SHAs (the 0.5.21 file equals his BEFORE)
        _g = SITE / "sglang/srt/layers/quantization/gptq/schemes/gptq_moe.py"
        _h = _sha(_g)
        if _h == D2_BEFORE:
            _t = _g.read_bytes()
            assert _t.count(b"dtype=torch.half") == 2
            _tmp = _g.with_name(_g.name + ".e13tmp"); _tmp.write_bytes(_t.replace(b"dtype=torch.half", b"dtype=params_dtype"))
            os.replace(_tmp, _g); _h = _sha(_g)
        if _h != D2_AFTER:
            raise E11Fail(f"patch #3 after-sha mismatch {_h[:12]}")
    stage("2_fa2_and_patch3", True)
except Exception as e:
    stage("2_fa2_and_patch3", False, error=f"{type(e).__name__}: {e}"[:1500])

def apply_p1():
    """PLAN D3 fallback: route sm89 decode to SGLang's own Triton packed-decode branch (the is_hip() branch)."""
    p = SITE / "sglang/srt/layers/attention/qwen_sparse_attn_backend.py"
    _replace_once_atomic(p, "from sglang.srt.utils import is_hip\n",
                         "from sglang.srt.utils import is_hip\nimport os as _e13_os\n"
                         "_E13_SM89_TRITON_DECODE = _e13_os.environ.get('E13_SM89_TRITON_DECODE') == '1'\n", "P1 import")
    _replace_once_atomic(p, "        if is_hip():\n", "        if is_hip() or _E13_SM89_TRITON_DECODE:\n", "P1 branch")

def apply_p2():
    """PLAN B2 option: SGLang's torch reference for the QSA indexer MQA instead of TileLang."""
    p = SITE / "sglang/srt/layers/attention/qsa/mqa.py"
    _replace_once_atomic(p, "    HAS_TILELANG = True\n",
                         "    HAS_TILELANG = __import__('os').environ.get('E13_DISABLE_TILELANG') != '1'\n", "P2")

# ---- static asserts (PLAN 3c): version, GPUs, FA2 ABI, patch, model config, flag existence
def static_asserts():
    info = {}
    probe = ("import json, torch, sglang; d={'sglang': sglang.__version__, 'torch': torch.__version__,"
             "'cxx11abi': torch.compiled_with_cxx11_abi(), 'caps': [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]};"
             "\ntry:\n import flash_attn; d['flash_attn']=flash_attn.__version__\nexcept Exception as e: d['flash_attn']='ERR '+str(e)[:200]"
             "\ntry:\n from sglang.srt.layers.attention.qwen_sparse_attn_backend import _resolve_flash_attn_varlen_func as r"
             "\n d['qsa_decode_resolves_to']=r().__module__\nexcept Exception as e: d['qsa_decode_resolves_to']='ERR '+str(e)[:300]"
             "\nprint(json.dumps(d))")
    if E13_HARNESS:
        info.update(sglang="0.5.21", caps=[[8, 9]] * 4, flash_attn="2.8.3.post1", cxx11abi=True, harness=True,
                    qsa_decode_resolves_to="flash_attn.flash_attn_interface")
    else:
        r = subprocess.run([str(VPY), "-c", probe], capture_output=True, text=True, env=VENV_ENV, timeout=900)
        info.update(json.loads(r.stdout.strip().splitlines()[-1]) if r.returncode == 0 else {"probe_error": r.stderr[-800:]})
    cfg = json.loads((MODEL_DIR / "config.json").read_text()) if MODEL_DIR else {}
    tc = cfg.get("text_config", cfg); qc = _qc(MODEL_DIR) if MODEL_DIR else {}
    info.update(desc_act=qc.get("desc_act"), quant=qc.get("quant_method"), bits=qc.get("bits"), group=qc.get("group_size"),
                indexer=(tc.get("indexer_budget"), tc.get("indexer_compress_ratio")), kv_heads=tc.get("num_key_value_heads"),
                moe_inter=tc.get("moe_intermediate_size"), experts=tc.get("num_experts"))
    # effective MoE shapes per rank (PLAN 3c Marlin): EP4 keeps whole experts -> w13 N = 2 x inter, w2 K = inter
    inter = tc.get("moe_intermediate_size") or 0
    info["moe_shapes_ep4"] = {"w13_N": 2 * inter, "w2_K": inter, "experts_per_rank": (tc.get("num_experts") or 0) // 4,
                              "multiple_of_group128": (2 * inter) % 128 == 0 and inter % 128 == 0}
    ok = (str(info.get("sglang", "")).startswith("0.5.21") and len(info.get("caps") or []) == 4
          and all(tuple(c) == (8, 9) for c in info["caps"]) and info.get("cxx11abi") is True
          and not info.get("desc_act") and info["indexer"] == (2048, 4)
          and info["moe_shapes_ep4"]["multiple_of_group128"]) or E13_HARNESS
    # FA2 version / resolver are recorded, not gated here: if FA2 is broken, the micro-preflight picks P1 (PLAN 1b)
    info["fa2_resolved"] = str(info.get("flash_attn", "")).startswith("2.8.3") and \
        str(info.get("qsa_decode_resolves_to", "")).startswith("flash_attn.") and "cute" not in str(info.get("qsa_decode_resolves_to"))
    return ok, info

try:
    _ok, _info = static_asserts()
    stage("3_static_asserts", _ok, **_info)
except Exception as e:
    stage("3_static_asserts", False, error=f"{type(e).__name__}: {e}"[:1000])

TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def run(cmd, *, check=True, env=None):
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout.strip(): print(result.stdout.strip()[-6000:])
    if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
    if check and result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
    return result


def find_unique(root, name, *, directory=False, required=True):
    root = Path(root)
    direct = root if root.name == name else root / name
    valid = lambda p: p.is_dir() if directory else p.is_file()
    if valid(direct): return direct
    hits = sorted(p for p in root.rglob(name) if valid(p))
    if len(hits) > 1:
        raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"{name} not found under {root}")
    return None


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
    return h.hexdigest()


def indexed_shards(root):
    root = Path(root)
    index = json.loads((root / "model.safetensors.index.json").read_text())
    names = sorted(set(index["weight_map"].values()))
    if not names: raise RuntimeError(f"Empty weight index: {root}")
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"Invalid shard path: {name}")
        p = root / rel
        if not p.is_file() or p.stat().st_size < 8:
            raise RuntimeError(f"Missing/empty shard: {p}")
    return index, names


def prepare_draft_view(target, source_root, work_root):
    """Adapt metadata only; link original dataset weights without copying them."""
    target, source_root, work_root = map(Path, (target, source_root, work_root))
    configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
    configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
    candidates = []
    for path in configs:
        cfg = json.loads(path.read_text())
        q = cfg.get("quantization_config", {})
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            candidates.append((path.parent, cfg))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
    source, cfg = candidates[0]
    index, shards = indexed_shards(source)
    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
        raise RuntimeError("Draft index does not contain the expected MTP experts")
    cfg = copy.deepcopy(cfg)
    q = cfg["quantization_config"]
    group = q["config_groups"]["mtp_routed_experts"]
    w = group["weights"]
    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
        raise RuntimeError("Expected symmetric INT4 group32 draft experts")
    if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
        raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
    if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
        raise RuntimeError("Unexpected draft ignore rules")
    group["targets"] = [EXPERT_TARGET]
    q["ignore"] = [DENSE_IGNORE]
    # Content/path-specific directory makes notebook reruns idempotent without
    # overwriting an unrelated view. We never modify the read-only datasets.
    identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
    suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
    view = work_root / f"draft-view-{suffix}"
    view.mkdir(parents=True, exist_ok=True)
    config_path = view / "config.json"
    config_text = json.dumps(cfg, indent=2) + "\n"
    if config_path.exists() or config_path.is_symlink():
        if config_path.is_symlink() or config_path.read_text() != config_text:
            raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
    else:
        config_path.write_text(config_text)
    links = {name: source / name for name in shards}
    links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
    for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
        if (target / name).is_file(): links[name] = target / name
    for name, src in links.items():
        dest = view / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() or dest.is_symlink():
            if not dest.is_symlink() or dest.resolve() != src.resolve():
                raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
        else:
            dest.symlink_to(src.resolve())
    print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
    return view




In [ ]:
# ==================================================================== CELL 3 — micro-preflight (GPU kernels, no model)
MICRO = {}
def _run_script(name, args, timeout):
    cmd = [str(VPY) if not E13_HARNESS else sys.executable, "-I", str(SCRIPT_DIR / name)] + args
    t = time.perf_counter()
    r = subprocess.run(cmd, capture_output=True, text=True, env=VENV_ENV if not E13_HARNESS else None, timeout=timeout)
    (OUT / f"{name}.log").write_text(r.stdout + "\n--- stderr ---\n" + r.stderr, encoding="utf-8")
    return r.returncode, round(time.perf_counter() - t, 1)

_pre_ok = all(REPORT["stages"].get(s, {}).get("ok") for s in ("1_install", "2_fa2_and_patch3", "3_static_asserts"))
if _pre_ok:
    _self = ["--cpu-selftest"] if E13_HARNESS else []
    for _name, _extra, _to in (("mp_decode.py", _self, 1800), ("mp_topk.py", _self + ["--fixtures", str(OUT / "topk_fixtures")], 2400),
                               ("mp_mqa.py", [], 1800),
                               ("mp_pinned.py", ["--model-dir", str(MODEL_DIR)] + (["--per-rank-bytes", "1000000"] if E13_HARNESS else []), 900)):
        if E13_HARNESS and _name in ("mp_mqa.py", "mp_pinned.py"):           # need a GPU; plumbing only locally
            MICRO[_name[3:-3]] = {"pass": True, "harness": True}; continue
        try:
            _rc, _sec = _run_script(_name, ["--out", str(OUT / (_name[:-3] + ".json"))] + _extra, _to)
            MICRO[_name[3:-3]] = json.loads((OUT / (_name[:-3] + ".json")).read_text())
            MICRO[_name[3:-3]]["_rc"], MICRO[_name[3:-3]]["_sec"] = _rc, _sec
        except Exception as e:
            MICRO[_name[3:-3]] = {"error": f"{type(e).__name__}: {e}"[:800]}
    _d = MICRO.get("decode", {})
    _g = {g.get("impl"): g.get("pass") for g in _d.get("graph", [])}
    fa2_ok = bool((_d.get("fa2") or {}).get("pass")) and (_g.get("fa2") is not False) \
        and bool(REPORT["stages"].get("3_static_asserts", {}).get("fa2_resolved"))      # the server must resolve to FA2 too
    p1_ok = bool((_d.get("p1") or {}).get("pass")) and (_g.get("p1_triton_packed") is not False)
    if E13_HARNESS:
        fa2_ok, p1_ok = True, False
    DECODE = "fa2" if fa2_ok else ("p1" if p1_ok else None)          # PLAN 4 1b pre-declared choice
    _t1 = (MICRO.get("topk", {}).get("t1") or {}).get("summary"); _t2 = (MICRO.get("topk", {}).get("t2") or {}).get("summary")
    REPORT["decision"]["micro"] = {"decode": DECODE, "fa2_pass": fa2_ok, "p1_pass": p1_ok, "graph": _g,
                                   "mqa_pass": MICRO.get("mqa", {}).get("pass"), "topk_t1": _t1, "topk_t2": _t2,
                                   "pinned_pass": MICRO.get("pinned", {}).get("pass")}
    MICRO_FLAGS = {"mqa_pass": MICRO.get("mqa", {}).get("pass"),
                   "decode_graph_pass": _g.get("fa2" if DECODE == "fa2" else "p1_triton_packed"),
                   "alt_decode_pass": p1_ok if DECODE == "fa2" else fa2_ok}
    stage("4_micro_preflight", DECODE is not None and MICRO.get("pinned", {}).get("pass") is True,
          **REPORT["decision"]["micro"])
else:
    DECODE, MICRO_FLAGS = None, {}
    stage("4_micro_preflight", False, error="skipped: install / patch / static asserts failed")


In [ ]:
# ==================================================================== CELL 4 — Launch 1 (target-only), gates, Launch 2
def e13_flags(cfg):
    gb = [n for n in sorted({1, 2, 4, 7, 8, 9, 10, cfg["running"]}) if n <= cfg["running"]]   # Franzen's graph sizes
    f = ["--model-path", str(MODEL_DIR), "--load-format", "safetensors",
         "--model-loader-extra-config", json.dumps({"enable_multithread_load": False}),
         "--served-model-name", SERVED, "--host", "127.0.0.1", "--port", str(SGL_PORT),
         "--tensor-parallel-size", "4", "--ep-size", "4",                                       # D4
         "--dtype", "bfloat16", "--quantization", "auto-round", "--kv-cache-dtype", "fp8_e4m3",
         "--mem-fraction-static", cfg["mem"], "--context-length", str(cfg["ctx"]), "--page-size", "64",
         "--max-running-requests", str(cfg["running"]), "--chunked-prefill-size", str(cfg["chunk"]),
         "--max-prefill-tokens", "16384",
         "--mamba-ssm-dtype", "bfloat16", "--max-mamba-cache-size", str(cfg["mamba_cache"]),
         "--mamba-radix-cache-strategy", "extra_buffer", "--mamba-track-interval", "64",
         "--mamba-backend", "triton", "--linear-attn-decode-backend", "triton", "--linear-attn-prefill-backend", "triton",  # D5
         "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
         "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
         "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
         "--default-chat-template-kwargs", json.dumps({"preserve_thinking": True}),
         "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
         "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging",
         "--weight-loader-prefetch-checkpoints", "--weight-loader-drop-cache-after-load"]
    if cfg.get("no_cuda_graph"):
        f += ["--disable-cuda-graph"]
    else:
        f += ["--cuda-graph-max-bs-decode", str(cfg["running"]), "--cuda-graph-bs-decode", *map(str, gb)]
    if MODEL_DIR and (MODEL_DIR / "chat_template.jinja").exists():
        f += ["--chat-template", str(MODEL_DIR / "chat_template.jinja")]
    if cfg.get("nextn"):
        f += ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", "3", "--speculative-eagle-topk", "1",
              "--speculative-num-draft-tokens", "4", "--speculative-draft-model-path", str(cfg["draft_view"]),
              "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
              "--speculative-draft-kv-cache-dtype", "fp8_e4m3", "--speculative-accept-threshold-single", "1.0",
              "--speculative-accept-threshold-acc", "1.0", "--speculative-token-map", str(cfg["token_map"])]
    for bad in ("--gdn-mtp-cache-mode", "--language-model-only"):
        assert bad not in f
    return f

def e13_env(cfg):
    env = dict(VENV_ENV or os.environ)
    env.update({"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
                "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false", "SGLANG_MM_PREPROCESS_DEVICE": "cpu",
                "OMP_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false", "TORCH_CUDA_ARCH_LIST": "8.9"})          # D6
    if cfg.get("decode") == "p1":
        env["E13_SM89_TRITON_DECODE"] = "1"
    if cfg.get("tilelang_off"):
        env["E13_DISABLE_TILELANG"] = "1"
    return env

class GpuSampler(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True); self.max_used = None; self.min_avail = None; self.stop_ = False
    def run(self):
        while not self.stop_:
            u = _e11_gpu_used()
            if u:
                self.max_used = max(u) if self.max_used is None else max(self.max_used, max(u))
            try:
                for line in open("/proc/meminfo"):
                    if line.startswith("MemAvailable"):
                        v = int(line.split()[1]) * 1024 / 1e9
                        self.min_avail = v if self.min_avail is None else min(self.min_avail, v)
            except Exception:
                pass
            time.sleep(2)

def wait_ready_progress(srv, cap):
    """/health only while the launcher lives; fail on 15 min of log silence (progress-based, R6)."""
    last_size, last_change = -1, time.perf_counter()
    while time.perf_counter() - srv.t0 < cap and left() > 60:
        if not srv.alive():
            return False, f"server exited rc={srv.proc.returncode}"
        try:
            _urlreq.urlopen(SGL_URL + "/health", timeout=10).read()
            return True, f"ready in {time.perf_counter() - srv.t0:.0f}s"
        except Exception:
            pass
        sz = srv.log_path.stat().st_size if srv.log_path.exists() else 0
        if sz != last_size:
            last_size, last_change = sz, time.perf_counter()
        elif time.perf_counter() - last_change > READY_STALL_SEC:
            return False, "stalled: no log progress for 15 min"
        time.sleep(5)
    return False, "timeout"

def run_launch(cfg, budget_sec):
    rec = {"cfg": {k: v for k, v in cfg.items() if k not in ("draft_view", "token_map")}, "t_start": round(elapsed())}
    t_launch = time.perf_counter()
    if cfg.get("decode") == "p1" and not E13_HARNESS:
        apply_p1()
    if cfg.get("tilelang_off") and not E13_HARNESS:
        apply_p2()
    rec["gpu_free_before"] = _e11_gpu_free(180)
    cmd = ([str(VPY), "-m", "sglang.launch_server"] if not E13_HARNESS else [sys.executable, os.environ["E13_FAKE_SERVER"]])
    flags = e13_flags(cfg); rec["flags"] = flags
    srv = E11Server(cmd + flags, e13_env(cfg), cfg["tag"]); srv.log_path = OUT / f"sglang_server_{cfg['tag']}.log"
    sampler = GpuSampler(); sampler.start()
    try:
        srv.start()
        ok, why = wait_ready_progress(srv, min(READY_CAP_SEC, budget_sec - 300))
        rec["ready"] = ok; rec["ready_note"] = why
        mem = e13_gate.memory_gate(srv.log_text()); rec["memory"] = mem
        if not ok:
            rec["gate_c"] = {"pass": False, "signature": "oom_or_cuda_error" if mem.get("oom") else "load_failed",
                             "hard_failures": [why]}
            rec["log_tail"] = srv.log_text()[-6000:]
            return rec
        gate = e13_gate.Gate(SGL_URL, SERVED, str(OUT), log=lambda m: print(m, flush=True))
        running = mem.get("capped_running") or cfg["running"]
        rec["gate_c"] = gate.run_gate_c(mem.get("kv_tokens"), running)
        rec["peak_gpu_used_mb_gate"] = sampler.max_used
        if rec["gate_c"]["pass"] and time.perf_counter() - t_launch < budget_sec - 900:
            kv = mem.get("kv_tokens") or 0
            levels = [n for n in (1, 2, 4, 8) if n <= running and n * 16600 <= max(kv, 1)] or [1]
            rec["speed_16k"] = gate.speed(levels, 16000)
            if kv >= 2 * 50000 and time.perf_counter() - t_launch < budget_sec - 600:
                rec["speed_48k"] = gate.speed([n for n in (1, 2) if n * 50000 <= kv], 48000)
        rec["peak_gpu_used_mb"] = sampler.max_used; rec["min_host_avail_gb"] = sampler.min_avail
        rec["peak_free_gb"] = round((23034 - sampler.max_used) / 1024, 2) if sampler.max_used else None   # L4 = 23034 MiB
        return rec
    except Exception as e:
        rec["error"] = traceback.format_exc()[-2000:]
        rec.setdefault("gate_c", {"pass": False, "signature": "exception", "hard_failures": [str(e)[:300]]})
        return rec
    finally:
        sampler.stop_ = True
        rec["memory_after"] = e13_gate.memory_gate(srv.log_text())       # the decision uses the log at the END (OOM later)
        if rec["memory_after"].get("oom") and rec.get("gate_c"):
            rec["gate_c"]["signature"] = "oom_or_cuda_error"
        rec["server_log_tail"] = srv.log_text()[-4000:]
        srv.kill(wait=True)
        rec["t_end"] = round(elapsed())
        REPORT["launches"].append(rec); write_json_atomic(OUT / "e13_probe_report.json", REPORT)

if REPORT["stages"].get("4_micro_preflight", {}).get("ok") and left() > L1_SEC:
    L1 = dict(LAUNCH1, decode=DECODE, tilelang_off=(MICRO_FLAGS.get("mqa_pass") is False))   # PLAN 1b: indexer rule
    r1 = run_launch(L1, L1_SEC)
    stage("5_launch1", bool(r1.get("gate_c", {}).get("pass")), ready=r1.get("ready"), note=r1.get("ready_note"),
          memory=r1.get("memory"), gate=r1.get("gate_c", {}).get("hard_failures"), signature=r1.get("gate_c", {}).get("signature"),
          speed=r1.get("speed_16k"))
    kind, why = e13_gate.launch2_decision(r1.get("gate_c", {}), r1.get("memory_after") or r1.get("memory", {}),
                                          r1.get("peak_free_gb"), MICRO_FLAGS)
    REPORT["decision"]["launch2"] = {"kind": kind, "why": why}
    stage("6_launch2_decision", True, kind=kind, why=why)
    L2 = None
    if kind == "nextn_on":
        try:
            dv = prepare_draft_view(MODEL_DIR, DRAFT_DIR, Path("/tmp/e13-draft"))
            tok = find_unique(PENNY_DIR, "hot_tokens_64k.pt")
            if sha256(tok) != TOKEN_MAP_SHA or sha256(MODEL_DIR / "tokenizer.json") != TOKENIZER_SHA:
                raise E11Fail("FR-Spec map or tokenizer sha differs from Franzen's")
            L2 = dict(L1, tag="L2_nextn", nextn=True, draft_view=dv, token_map=tok)
        except Exception as e:
            stage("6b_draft_view", False, error=f"{type(e).__name__}: {e}"[:800])
    elif kind == "chunk_4096":
        L2 = dict(L1, tag="L2_chunk4096", chunk=4096)
    elif kind == "b2_tilelang_off":
        L2 = dict(L1, tag="L2_b2_tilelang_off", tilelang_off=True)
    elif kind == "b2_no_cuda_graph":
        L2 = dict(L1, tag="L2_b2_no_graph", no_cuda_graph=True)
    elif kind == "b2_other_decode":
        L2 = dict(L1, tag="L2_b2_decode", decode=("p1" if DECODE == "fa2" else "fa2"))
    elif kind == "b2_diagnostic":
        L2 = dict(L1, tag="L2_b2_diagnostic", tilelang_off=True, no_cuda_graph=True)
    if L2 is not None and left() > L2_SEC:
        r2 = run_launch(L2, L2_SEC)
        stage("7_launch2", bool(r2.get("gate_c", {}).get("pass")), tag=L2["tag"], ready=r2.get("ready"),
              memory=r2.get("memory"), gate=r2.get("gate_c", {}).get("hard_failures"), speed=r2.get("speed_16k"))
    elif L2 is not None:
        stage("7_launch2", False, error=f"not enough time left ({left():.0f}s)")
else:
    stage("5_launch1", False, error="skipped: micro-preflight failed or not enough time")


In [ ]:
# ==================================================================== CELL 5 — report and bundle
REPORT["elapsed_sec"] = round(elapsed()); REPORT["script_sha"] = {"mp_topk.py": "2fbd90364c8ff710", "mp_decode.py": "0bd5bfe146f54cf5", "mp_mqa.py": "377379f60d711c40", "mp_pinned.py": "2a09f0efdeeba399", "e13_gate.py": "015f9d4189c42700"}
write_json_atomic(OUT / "e13_probe_report.json", REPORT)
with zipfile.ZipFile(WORKING / "E13_PROBE_BUNDLE.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and p.stat().st_size < 200 * 1024 * 1024:
            z.write(p, p.relative_to(WORKING))
print("E13 PROBE REPORT:", OUT / "e13_probe_report.json", "| elapsed", REPORT["elapsed_sec"], "s")
for k, v in REPORT["stages"].items():
    print(f"  {k:22s} {'OK' if v['ok'] else 'FAIL'}")
print("decision:", json.dumps(REPORT["decision"], default=str)[:2000])
